# Email Phishing Detection — Walkthrough

CSC4850 Machine Learning (Summer 2026). Trains and compares Naive Bayes,
Logistic Regression, and a linear SVM on the CEAS_08 phishing dataset.

Make sure `data/CEAS_08.csv` exists (see `data/README.md`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)

DATA_PATH = "../data/CEAS_08.csv"
RANDOM_STATE = 42

## 1. Load and clean\n\nFill missing subject/body **before** combining, then drop empty and duplicate emails.

In [ ]:
df = pd.read_csv(DATA_PATH)
df['subject'] = df['subject'].fillna('')
df['body'] = df['body'].fillna('')
df['subject_and_body'] = (df['subject'] + ' ' + df['body']).str.strip()
df = df.drop(columns=['subject', 'body'])
df = df[df['subject_and_body'] != ''].drop_duplicates('subject_and_body').reset_index(drop=True)
print(f"{len(df):,} emails after cleaning ({df['label'].mean():.1%} phishing)")
df.head()

## 2. TF-IDF features

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(df['subject_and_body'])   # sparse - do not densify
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 3. Train and compare the three models

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM': LinearSVC(),
}
rows, fitted, scores = {}, {}, {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    fitted[name] = clf
    pred = clf.predict(X_test)
    scores[name] = clf.predict_proba(X_test)[:, 1] if hasattr(clf, 'predict_proba') \
                   else clf.decision_function(X_test)
    rows[name] = {
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, scores[name]),
    }
results = pd.DataFrame(rows).T.round(4)
results

## 4. Visualize

In [ ]:
plt.figure(figsize=(6, 5))
for name in models:
    fpr, tpr, _ = roc_curve(y_test, scores[name])
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={results.loc[name,'ROC-AUC']:.4f})")
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves'); plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, clf) in zip(axes, fitted.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, clf.predict(X_test)),
        display_labels=['Legit', 'Phish']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)
plt.tight_layout(); plt.show()

## 5. Takeaways

- The **linear SVM** performs best overall; all three exceed 98% F1.
- Train vs. test accuracy are nearly identical -> little overfitting.
- **Limitation:** legitimate emails skew toward tech mailing lists, so these
  scores may not transfer to a general inbox.